# BÀI THỰC HÀNH 2: MẠNG NEURAL TÍCH CHẬP

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses..

Bộ dữ liệu sử dụng: [VinaFood21 dataset](https://arxiv.org/abs/2108.02929).

Link download: https://drive.google.com/file/d/1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR/view?usp=share_link.

In [5]:
pip install torch torchvision transformers scikit-learn matplotlib seaborn

In [6]:
# Import thư viện và cố định seed
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch  :", torch.__version__)
print("device :", device)


torch  : 2.11.0+cu130
device : cuda


In [8]:
# Hai hàm tiện ích dùng lại trong cả notebook
def n_params(module, trainable_only=False):
    ps = module.parameters()
    if trainable_only:
        ps = (p for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in ps)


def show(label, t):
    print(f"{label:<28} {tuple(t.shape)}")

# Tải và giải nén bộ dữ liệu VinaFood21 từ Google Drive
import os
import zipfile
import gdown

file_id = '1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR'
output = 'VinaFood21.zip'
url = f'https://drive.google.com/uc?id={file_id}'

if not os.path.exists(output) and not os.path.exists('VinaFood21'):
    print("Đang tải dữ liệu VinaFood21 từ Google Drive...")
    gdown.download(url, output, quiet=False)

if os.path.exists(output) and not os.path.exists('VinaFood21'):
    print("Đang giải nén dữ liệu...")
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Giải nén thành công!")
else:
    print("Bộ dữ liệu đã tồn tại sẵn trong hệ thống.")

Đang tải dữ liệu VinaFood21 từ Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR
From (redirected): https://drive.google.com/uc?id=1UpZOf0XlwvB4rKpyZ35iwTA8oWHqDBbR&confirm=t&uuid=c21c4b5e-e937-4dbf-a22b-2787947e918e
To: /content/VinaFood21.zip
100%|██████████| 2.71G/2.71G [00:39<00:00, 68.0MB/s]


Đang giải nén dữ liệu...
Giải nén thành công!


In [9]:
# Cấu hình chung cho cả bốn bài: đường dẫn, kích thước ảnh, batch size, seed
from pathlib import Path

DATA_ROOT = Path("VinaFood21")        # sửa đường dẫn cho đúng máy
IMG_SIZE = 64
BATCH = 32
SEED = 42

HAS_DATA = (DATA_ROOT / "train").is_dir()
print("Tìm thấy dữ liệu:", HAS_DATA, "-", DATA_ROOT.resolve())


Tìm thấy dữ liệu: True - /content/VinaFood21


In [4]:
from torchvision import transforms

# Kiểm tra xem torchvision đã được import thành công chưa
HAS_TV = True

# Khai báo MEAN và STD (thường dùng bộ chỉ số của ImageNet)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Hai bộ transform riêng: train có augment, val/test không
if HAS_TV:
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    print("Đã dựng train_tf và eval_tf.")

Đã dựng train_tf và eval_tf.


In [12]:
# Tạo dataset và DataLoader
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import torch

CLASSES = [f"lop_{i}" for i in range(21)]
train_ds = val_ds = train_dl = val_dl = None

if HAS_DATA:
    # Load toàn bộ dữ liệu từ thư mục train
    full_dataset = datasets.ImageFolder(DATA_ROOT / "train")
    CLASSES = full_dataset.classes

    # Chia tập dữ liệu thành 80% train và 20% val
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Thiết lập seed để việc chia dữ liệu được cố định
    generator = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=generator)

    # Áp dụng các phép biến đổi (transforms) riêng biệt
    train_ds.dataset.transform = train_tf
    # Để tránh việc val_ds dùng chung transform bị augment với train_ds,
    # ta gán eval_tf cho tập validation
    val_ds.dataset.transform = eval_tf

    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)

    print("số lớp          :", len(CLASSES))
    print("số ảnh train   :", len(train_ds))
    print("số ảnh val     :", len(val_ds))
    print("danh sách lớp   :", CLASSES[:5], "...")
else:
    print("Bỏ qua phần load dữ liệu.")

số lớp          : 21
số ảnh train   : 8035
số ảnh val     : 2009
danh sách lớp   : ['banh-can', 'banh-hoi', 'banh-mi-chao', 'banh-tet', 'banh-trang-tron'] ...


In [14]:
# Chiều tensor trước và sau DataLoader: một ảnh so với một batch
READY = True

if READY:
    anh, nhan = train_ds[0]
    xb, yb = next(iter(train_dl))
    show("một ảnh sau transform", anh)
    show("một batch vào model", xb)
    show("nhãn", yb)
else:
    show("một ảnh sau transform", torch.rand(3, IMG_SIZE, IMG_SIZE))
    show("một batch vào model", torch.rand(BATCH, 3, IMG_SIZE, IMG_SIZE))
    print("(tensor giả lập, vì chưa có dữ liệu thật)")

một ảnh sau transform        (3, 64, 64)
một batch vào model          (32, 3, 64, 64)
nhãn                         (32,)


### Bài 1: Xây dựng mô hình LeNet. Huấn luyện và đánh giá mô hình LeNet trên 4 độ đo precision, recall và F1-macro (sử dụng Adam làm optimizer).

![image.png](attachment:image.png)

In [15]:
# Định nghĩa LeNet-5 với số channel vào, kích thước ảnh và số lớp tham số hoá
class LeNet5(nn.Module):
    def __init__(self, in_ch=3, img=64, num_classes=21, pool="avg"):
        super().__init__()
        Pool = nn.AvgPool2d if pool == "avg" else nn.MaxPool2d
        self.c1 = nn.Conv2d(in_ch, 6, kernel_size=5)
        self.s2 = Pool(2, 2)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        self.s4 = Pool(2, 2)

        n_flat = self._flat_dim(in_ch, img)
        self.f5 = nn.Linear(n_flat, 120)
        self.f6 = nn.Linear(120, 84)
        self.out = nn.Linear(84, num_classes)
        self.n_flat = n_flat

    def _flat_dim(self, in_ch, img):
        # Dò số chiều sau S4 bằng một forward giả, tránh tính tay
        with torch.no_grad():
            z = torch.zeros(1, in_ch, img, img)
            z = self.s4(self.c3(self.s2(self.c1(z))))
        return int(z.flatten(1).shape[1])

    def forward(self, x):
        x = self.s2(torch.relu(self.c1(x)))
        x = self.s4(torch.relu(self.c3(x)))
        x = x.flatten(1)
        x = torch.relu(self.f5(x))
        x = torch.relu(self.f6(x))
        return self.out(x)


In [16]:
# Khởi tạo cấu hình của Bài 1: ảnh 64x64x3, 21 lớp
lenet = LeNet5(in_ch=3, img=64, num_classes=21)
print("số chiều sau flatten:", lenet.n_flat)
print("tổng tham số        :", f"{n_params(lenet):,}")

assert lenet.n_flat == 2704
assert n_params(lenet) == 339_421


số chiều sau flatten: 2704
tổng tham số        : 339,421


In [17]:
# Hàm dựng bảng shape và tham số của từng layer
def layer_table(model, img, in_ch=3):
    x = torch.zeros(1, in_ch, img, img)
    rows, total = [], 0
    for name, layer in [("C1", model.c1), ("S2", model.s2),
                        ("C3", model.c3), ("S4", model.s4)]:
        x = layer(x)
        p = n_params(layer)
        total += p
        rows.append((name, "x".join(map(str, tuple(x.shape)[1:])), p))
    x = x.flatten(1)
    rows.append(("flatten", str(x.shape[1]), 0))
    for name, layer in [("F5", model.f5), ("F6", model.f6), ("OUT", model.out)]:
        x = layer(x)
        p = n_params(layer)
        total += p
        rows.append((name, str(x.shape[1]), p))
    return rows, total
# In bảng cho cấu hình ảnh 64x64
rows, total = layer_table(lenet, 64)

print(f"{'layer':<10}{'output':<16}{'tham số':>12}")
for name, shape, p in rows:
    print(f"{name:<10}{shape:<16}{p:>12,}")
print(f"{'TỔNG':<10}{'':<16}{total:>12,}")
assert total == 339_421

# Nhận xét: tham số dồn gần hết vào F5, hai conv layer rất nhẹ


layer     output               tham số
C1        6x60x60                  456
S2        6x30x30                    0
C3        16x26x26               2,416
S4        16x13x13                   0
flatten   2704                       0
F5        120                  324,600
F6        84                    10,164
OUT       21                     1,785
TỔNG                           339,421


In [18]:
# So sánh cấu hình ảnh 32x32 và 64x64
print(f"{'resize':<10}{'sau S4':<14}{'flatten':>9}{'F5':>12}{'tổng':>12}{'conv %':>9}")
for img in (32, 64):
    m = LeNet5(in_ch=3, img=img, num_classes=21)
    conv_p = n_params(m.c1) + n_params(m.c3)
    side = int((m.n_flat // 16) ** 0.5)
    print(f"{img}x{img:<5}{f'{side}x{side}x16':<14}{m.n_flat:>9,}"
          f"{n_params(m.f5):>12,}{n_params(m):>12,}"
          f"{100 * conv_p / n_params(m):>8.2f}%")


resize    sau S4          flatten          F5        tổng   conv %
32x32   5x5x16              400      48,120      62,941    4.56%
64x64   13x13x16          2,704     324,600     339,421    0.85%


In [19]:
# Kiểm tra: phần conv giữ nguyên, chỉ phần fully connected phình ra
m32, m64 = LeNet5(img=32), LeNet5(img=64)

assert n_params(m32) == 62_941 and n_params(m64) == 339_421
assert n_params(m32.c1) + n_params(m32.c3) == 2872
assert n_params(m64.c1) + n_params(m64.c3) == 2872

print(f"Tổng tham số gấp {n_params(m64) / n_params(m32):.2f} lần,"
      " hai conv layer vẫn 2,872.")


Tổng tham số gấp 5.39 lần, hai conv layer vẫn 2,872.


In [20]:
# BatchNorm2d có 2 tham số học được cho mỗi channel (gamma, beta)
bn = nn.BatchNorm2d(64)
print("BatchNorm2d(64):", n_params(bn), "tham số = 2 * 64")
assert n_params(bn) == 128


BatchNorm2d(64): 128 tham số = 2 * 64


In [21]:
# Dropout cho kết quả khác nhau ở train và giống nhau ở eval
net = nn.Sequential(nn.Linear(10, 10), nn.Dropout(p=0.5))
x = torch.ones(4, 10)

net.train()
a, b = net(x), net(x)
print("model.train(): hai lần forward giống nhau?", torch.equal(a, b))

net.eval()
a, b = net(x), net(x)
print("model.eval() : hai lần forward giống nhau?", torch.equal(a, b))

# Nhận xét: quên eval() thì mỗi lần đánh giá ra một con số khác mà không báo lỗi


model.train(): hai lần forward giống nhau? False
model.eval() : hai lần forward giống nhau? True


In [23]:
# 1. Định nghĩa các hàm huấn luyện và đánh giá với các độ đo Precision, Recall, F1-macro
import torch.optim as optim
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_targets = []
    val_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(yb.cpu().numpy())

    val_loss /= len(dataloader.dataset)
    accuracy = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    return val_loss, accuracy, precision, recall, f1

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, device=device):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"Bắt đầu huấn luyện mô hình trên {device}...")
    print(f"{'Epoch':<8}{'Train Loss':<12}{'Val Loss':<12}{'Accuracy':<10}{'Precision':<11}{'Recall':<10}{'F1-Macro':<10}")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss, acc, prec, rec, f1 = evaluate_model(model, val_loader, device)

        print(f"{epoch:<8}{train_loss:<12.4f}{val_loss:<12.4f}{acc:<10.4f}{prec:<11.4f}{rec:<10.4f}{f1:<10.4f}")

# 2. Huấn luyện và đánh giá mô hình LeNet (Bài 1)
print("--- HUẤN LUYỆN LENET (BÀI 1) ---")
train_model(lenet, train_dl, val_dl, epochs=5, lr=1e-3, device=device)

--- HUẤN LUYỆN LENET (BÀI 1) ---
Bắt đầu huấn luyện mô hình trên cuda...
Epoch   Train Loss  Val Loss    Accuracy  Precision  Recall    F1-Macro  


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1       2.7796      2.6319      0.2041    0.1377     0.1462    0.1124    


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


2       2.5461      2.5024      0.2479    0.2171     0.2138    0.2018    


...

3       2.3848      2.4600      0.2474    0.2716     0.2059    0.2021    


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


4       2.2088      2.3992      0.2683    0.2801     0.2408    0.2389    


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


5       2.0042      2.3620      0.3051    0.2936     0.2889    0.2790    


### Bài 2: Xây dựng mô hình GoogLeNet. Huấn luyện và đánh giá mô hình GoogLeNet trên 4 độ đo precision, recall và F1 (sử dụng Adam làm optimizer).

**Inception Blocks**

![image.png](attachment:image.png)

**GoogLeNet (Multi-branch Network)**

![image-2.png](attachment:image-2.png)

In [ ]:
# Định nghĩa khối Inception của GoogLeNet
class Inception(nn.Module):
    def __init__(self, in_planes, n1x1, n3x3red, n3x3, n5x5red, n5x5, pool_planes):
        super(Inception, self).__init__()
        # Nhánh 1: 1x1 conv
        self.b1 = nn.Sequential(
            nn.Conv2d(in_planes, n1x1, kernel_size=1),
            nn.BatchNorm2d(n1x1),
            nn.ReLU(True),
        )
        # Nhánh 2: 1x1 conv -> 3x3 conv
        self.b2 = nn.Sequential(
            nn.Conv2d(in_planes, n3x3red, kernel_size=1),
            nn.BatchNorm2d(n3x3red),
            nn.ReLU(True),
            nn.Conv2d(n3x3red, n3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(n3x3),
            nn.ReLU(True),
        )
        # Nhánh 3: 1x1 conv -> 5x5 conv (ở đây dùng padding=2 để giữ nguyên kích thước)
        self.b3 = nn.Sequential(
            nn.Conv2d(in_planes, n5x5red, kernel_size=1),
            nn.BatchNorm2d(n5x5red),
            nn.ReLU(True),
            nn.Conv2d(n5x5red, n5x5, kernel_size=5, padding=2),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(True),
        )
        # Nhánh 4: 3x3 max pooling -> 1x1 conv
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_planes, pool_planes, kernel_size=1),
            nn.BatchNorm2d(pool_planes),
            nn.ReLU(True),
        )

    def forward(self, x):
        y1 = self.b1(x)
        y2 = self.b2(x)
        y3 = self.b3(x)
        y4 = self.b4(x)
        return torch.cat([y1, y2, y3, y4], 1)

# Định nghĩa mô hình GoogLeNet thu nhỏ phù hợp kích thước 64x64
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=21):
        super(GoogLeNet, self).__init__()
        self.pre_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
        )

        self.a3 = Inception(64, 32, 48, 64, 8, 16, 16)   # output channels: 32+64+16+16 = 128
        self.b3 = Inception(128, 64, 64, 120, 16, 32, 32) # output channels: 64+120+32+32 = 248

        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        self.a4 = Inception(248, 128, 128, 192, 32, 64, 64) # output: 128+192+64+64 = 448
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(448, num_classes)

    def forward(self, x):
        out = self.pre_layers(x)
        out = self.a3(out)
        out = self.b3(out)
        out = self.maxpool(out)
        out = self.a4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

# Khởi tạo GoogLeNet
googlenet = GoogLeNet(num_classes=21)
print("Tổng số tham số của GoogLeNet:", f"{n_params(googlenet):,}")

# 3. Tiến hành huấn luyện và đánh giá GoogLeNet (Bài 2)
print("\n--- HUẤN LUYỆN GOOGLENET (BÀI 2) ---")
train_model(googlenet, train_dl, val_dl, epochs=5, lr=1e-3, device=device)

Tổng số tham số của GoogLeNet: 516,341

--- HUẤN LUYỆN GOOGLENET (BÀI 2) ---
Bắt đầu huấn luyện mô hình trên cuda...
Epoch   Train Loss  Val Loss    Accuracy  Precision  Recall    F1-Macro  


### Bài 3*: Xây dựng mô hình ResNet-18, đánh giá mô hình ResNet-18 trên bộ dữ liệu VinaFood21 sử dụng các độ đo precision, recall, và F1 (Sử dụng Adam làm optimizer).

**ResNet Block**

![image.png](attachment:image.png)

**ResNet**

![image-2.png](attachment:image-2.png)

### Bài 4*: Sử dụng pretrained ResNet50 từ HuggingFace để fine-tune trên bộ dữ liệu VinaFood21.